In [0]:
import dlt
from pyspark.sql.types import StructType,StructField,IntegerType,DoubleType,StringType,TimestampType
import pyspark.sql.functions as F

In [0]:
input_schema_raw=StructType([StructField('CRASH DATE', StringType(), True), StructField('CRASH TIME', StringType(), True), StructField('BOROUGH', StringType(), True), StructField('ZIP CODE', StringType(), True), StructField('LATITUDE', DoubleType(), True), StructField('LONGITUDE', DoubleType(), True), StructField('LOCATION', StringType(), True), StructField('ON STREET NAME', StringType(), True), StructField('CROSS STREET NAME', StringType(), True), StructField('OFF STREET NAME', StringType(), True), StructField('NUMBER OF PERSONS INJURED', StringType(), True), StructField('NUMBER OF PERSONS KILLED', IntegerType(), True), StructField('NUMBER OF PEDESTRIANS INJURED', IntegerType(), True), StructField('NUMBER OF PEDESTRIANS KILLED', IntegerType(), True), StructField('NUMBER OF CYCLIST INJURED', IntegerType(), True), StructField('NUMBER OF CYCLIST KILLED', StringType(), True), StructField('NUMBER OF MOTORIST INJURED', StringType(), True), StructField('NUMBER OF MOTORIST KILLED', IntegerType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 1', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 2', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 3', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 4', StringType(), True), StructField('CONTRIBUTING FACTOR VEHICLE 5', StringType(), True), StructField('COLLISION ID', IntegerType(), True), StructField('VEHICLE TYPE CODE 1', StringType(), True), StructField('VEHICLE TYPE CODE 2', StringType(), True), StructField('VEHICLE TYPE CODE 3', StringType(), True), StructField('VEHICLE TYPE CODE 4', StringType(), True), StructField('VEHICLE TYPE CODE 5', StringType(), True)])

In [0]:
# file_path='/Volumes/learn_adb_fikrat/bronze/landing/crash-data/vehicle_collisions/'
# @dlt.table(table_properties={'quality': 'bronze', 'delta.columnMapping.mode': 'name',
#    'delta.minReaderVersion' : '3',   'delta.minWriterVersion' : '7'})

# def vehicle_accidents_batch():
#   return spark.read.format('csv').option('header','true').schema(input_schema_raw)\
#      .option("recursiveFileLookup", "true")\
#      .load(file_path)\
#      .withColumn('Bronze_Ingestion_Timestamp',F.current_timestamp())


In [0]:
file_path= spark.conf.get('source_path')
@dlt.table(name='vehicle_accidents_batch',
           table_properties={'quality': 'bronze', 'delta.columnMapping.mode': 'name',
   'delta.minReaderVersion' : '3',   'delta.minWriterVersion' : '7'})

def bronze_vehicle_crashes():
  return spark.read.format('csv').option('header','true').schema(input_schema_raw)\
     .option("recursiveFileLookup", "true")\
     .load(file_path)\
     .withColumn('Bronze_Ingestion_Timestamp',F.current_timestamp())


In [0]:
file_path='/Volumes/learn_adb_fikrat/bronze/landing/crash-data/vehicle_collisions/'
@dlt.view(name='vw_vehicle_accidents')
def vw_bronze_vehicle_crashes():
  return spark.read.format('csv').option('header','true').schema(input_schema_raw)\
     .option("recursiveFileLookup", "true")\
     .load(file_path)\
     .withColumn('Bronze_Ingestion_Timestamp',F.current_timestamp())


In [0]:
input_schema_silver=StructType([StructField('CRASH_DATE', StringType(), True), StructField('CRASH_TIME', StringType(), True), StructField('BOROUGH', StringType(), True), StructField('ZIP_CODE', StringType(), True), StructField('LATITUDE', DoubleType(), True), StructField('LONGITUDE', DoubleType(), True), StructField('LOCATION', StringType(), True), StructField('ON_STREET_NAME', StringType(), True), StructField('CROSS_STREET_NAME', StringType(), True), StructField('OFF_STREET_NAME', StringType(), True), StructField('NUMBER_OF_PERSONS_INJURED', StringType(), True), StructField('NUMBER_OF_PERSONS_KILLED', IntegerType(), True), StructField('NUMBER_OF_PEDESTRIANS_INJURED', IntegerType(), True), StructField('NUMBER_OF_PEDESTRIANS_KILLED', IntegerType(), True), StructField('NUMBER_OF_CYCLIST_INJURED', IntegerType(), True), StructField('NUMBER_OF_CYCLIST_KILLED', StringType(), True), StructField('NUMBER_OF_MOTORIST_INJURED', StringType(), True), StructField('NUMBER_OF_MOTORIST_KILLED', IntegerType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_1', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_2', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_3', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_4', StringType(), True), StructField('CONTRIBUTING_FACTOR_VEHICLE_5', StringType(), True), StructField('COLLISION_ID', IntegerType(), True), StructField('VEHICLE_TYPE_CODE_1', StringType(), True), StructField('VEHICLE_TYPE_CODE_2', StringType(), True), StructField('VEHICLE_TYPE_CODE_3', StringType(), True), StructField('VEHICLE_TYPE_CODE_4', StringType(), True), StructField('VEHICLE_TYPE_CODE_5', StringType(), True),
 StructField('Bronze_Ingestion_Timestamp', TimestampType(), True)])

Exploring Table properties

In [0]:
@dlt.table(name='silver.vehicle_accidents_cleansed_batch',table_properties={'data_zone': 'silver'})
def vehicle_accidents_cleansed_batch():
    df = dlt.read('vehicle_accidents_batch')
    for clm in df.schema:
        col_name=clm.name
        col_new_name=col_name.replace(' ', '_')
        col_type=input_schema_silver[col_new_name].dataType
        df = df.withColumn(col_new_name,F.col(col_name).cast(col_type))
        if col_name!= col_new_name:
            df=df.drop(col_name)
    df=df.withColumn('ACCIDENT_DATE_TIME', F.to_timestamp(\
        F.concat(F.col('CRASH_DATE'), F.lit(' '), F.lpad(F.col('CRASH_TIME'), 5, '0')), 'MM/dd/yyyy HH:mm'))\
        .drop('LATITUDE','LONGITUDE','CRASH_DATE','CRASH_TIME')\
        .filter(F.col('ACCIDENT_DATE_TIME').isNotNull())    
    
    return df

In [0]:
@dlt.table(table_properties={'data_zone': 'silver'})
def vehicle_accidents_cleansed_batch2():
    df = dlt.read('vw_vehicle_accidents')
    for clm in df.schema:
        col_name=clm.name
        col_new_name=col_name.replace(' ', '_')
        col_type=input_schema_silver[col_new_name].dataType
        df = df.withColumn(col_new_name,F.col(col_name).cast(col_type))
        if col_name!= col_new_name:
            df=df.drop(col_name)
    df=df.withColumn('ACCIDENT_DATE_TIME', F.to_timestamp(\
        F.concat(F.col('CRASH_DATE'), F.lit(' '), F.lpad(F.col('CRASH_TIME'), 5, '0')), 'MM/dd/yyyy HH:mm'))\
        .drop('LATITUDE','LONGITUDE','CRASH_DATE','CRASH_TIME')\
        .filter(F.col('ACCIDENT_DATE_TIME').isNotNull())    
    
    return df

###Quality Assurance

In [0]:
@dlt.table(name='accidents_qa_warn')
@dlt.expect('Valid borough','BOROUGH IS NOT NULL')
def accidents_qa_passed():
    return dlt.read('silver.vehicle_accidents_cleansed_batch')

In [0]:
@dlt.table(name='accidents_qa_drop')
@dlt.expect_or_drop('Valid borough','BOROUGH IS NOT NULL')
def accidents_qa_passed():
    return dlt.read('silver.vehicle_accidents_cleansed_batch')


In [0]:
# @dlt.table(name='accidents_qa_fail')
# @dlt.expect_or_fail('Valid borough','BOROUGH IS NOT NULL')
# def accidents_qa_passed():
#     return dlt.read('silver.vehicle_accidents_cleansed_batch')

In [0]:
qa_conditions = {"Valid borough":"BOROUGH IS NOT NULL","Passenger vehicle":"VEHICLE_TYPE_CODE_1 ='PASSENGER VEHICLE'"}
@dlt.table(name='accidents_qa_multi_condition_drop')
@dlt.expect_all_or_drop(qa_conditions)
def accidents_qa_passed():
    return dlt.read('silver.vehicle_accidents_cleansed_batch')


In [0]:
qa_conditions = {"Valid borough":"BOROUGH IS NOT NULL","Passenger vehicle":"VEHICLE_TYPE_CODE_1 ='PASSENGER VEHICLE'"}
qa_quaranteened_conditions = f"NOT({' AND '.join(qa_conditions.values())})"
@dlt.table(name='accidents_qa_multi_condition_quaranteened')
@dlt.expect_or_drop('Quaranteened',qa_quaranteened_conditions)
def accidents_qa_passed():
    return dlt.read('silver.vehicle_accidents_cleansed_batch')


In [0]:
# @dlt.table(name='Accident_locations')
# @dlt.expect_or_drop('Valid Borough','BOROUGH IS NOT NULL AND BOROUGH NOT IN (SELECT BOROUGH FROM bronze.crash_locations)') 
# def crash_locations():
#    return dlt.read('vehicle_accidents_cleansed_batch')\
#        .selectExpr('BOROUGH','LOCATION','ZIP_CODE')

In [0]:
# @dlt.table(name='Accident_locations',table_properties={'quality': 'silver'})
# def crash_locations():
#    return dlt.read('vehicle_accidents_cleansed')\
#        .withColumn('LOCATION_ID',F.monotonically_increasing_id())\
#        .selectExpr('LOCATION_ID','LOCATION','ZIP_CODE','BOROUGH','ON_STREET_NAME','OFF_STREET_NAME','CROSS_STREET_NAME')       

1. Table name specification- using function name vs 'name property 
2. Hardcoded config Parameterized pipeline
3. Views vs tables
4. Writing to different schema
5. Streaming vs batch
6. Continious vs triggered. How continious behaves with batch (MV)
7. QA Expectations- demo 3 types
8. Multi-expect
9. Quaranteened logic
10. Monitoring- publishing logs table  
